# Exercise 3: A return reader, test first

**Goal:** After this exercise, you can have Codex write a reader of three Form 1040 values test first, with a test for a missing element.

**Scenario:** Week 3 reads many returns, and the course's schema lets a return leave out its taxable interest. You read three values in the sample return yourself. Codex writes the tests from them first, one for a return with no taxable interest, and then the reader.

**Coding agent:** yes. You use Codex in a terminal.

**Where you work:** the folder `week-02/day-03/exercise-03-a-return-reader-test-first/lab` in the course folder. It is a Git repository of its own. You run the cells of this notebook and some commands in a terminal.

**Done when:** cell 3 prints `Done:`.

**Optional part:** none. Exercise 4 is the extra exercise.

**What you have:** in `lab`:

- `returns/sample-return.xml`, the sample return of exercise 2.
- `form1040.py`, with the function `read_form_1040`. Its docstring says what it returns. Its code is not written yet.
- `schema/return.xsd`, the course's schema for its returns: which elements a return must hold, and which it may leave out.
- `tests/`, with no test yet, and `AGENTS.md`, which says how to run the tests.

**Start from:** step 1. To start again, run the reset command of step 1 again. It deletes your commits in `lab`.

### Step 1. See the start

Open a terminal (**Terminal > New Terminal**). It opens in the course folder. Run:

```sh
source .venv/bin/activate
python3 reset.py exercise-03-a-return-reader-test-first
```

Then run cell 1. You should see `NotImplementedError: read_form_1040 is not written yet`, then `no tests ran`, then `Skills: none`.

In [ ]:
# 1. See the start. You do not need to read this code.
import subprocess
import sys
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if not (lab / ".git").is_dir():
    print("lab is not there yet. Run the two commands of step 1 in a terminal.")
else:
    call = subprocess.run([sys.executable, "-c", "from form1040 import read_form_1040; read_form_1040('returns/sample-return.xml')"],
                          cwd=lab, capture_output=True, text=True)
    print((call.stderr.strip().splitlines() or ["read_form_1040 returned without an error"])[-1])
    tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                           cwd=lab, capture_output=True, text=True)
    print(tests.stdout.strip().splitlines()[-1])
    print("Skills:", ", ".join(p.name for p in (lab / ".agents" / "skills").glob("test-*")) or "none")

### Step 2. Install the skill

In the terminal, run these two commands:

```sh
cd week-02/day-03/exercise-03-a-return-reader-test-first/lab
npx --yes skills@1.7.4 add https://github.com/openai/plugins/tree/main/plugins/superpowers/skills/test-driven-development --agent codex --yes
```

It is the install command of Tuesday's exercise 3. It copies the skill's folder from GitHub into `.agents/skills/test-driven-development` in `lab`, where Codex finds it. You should see `Installed 1 skill` and `Done!`. If the terminal says `npx: command not found`, tell your trainer: the workspace needs Node.js.

### Step 3. Read the expected values, then have Codex write the reader test first

First, find the expected values yourself. Open `lab/returns/sample-return.xml` in the editor (in the Explorer: `week-02 > day-03 > exercise-03-a-return-reader-test-first > lab > returns`). Inside the `IRS1040` element, find the values of `WagesAmt`, `TaxableInterestAmt` and `AdjustedGrossIncomeAmt`. Then open `lab/schema/return.xsd` and find `TaxableInterestAmt`. Its `minOccurs="0"` says a return may leave it out.

In the same terminal, run `codex`. If Codex asks if you trust this folder, answer yes. Write your request in your own words. It must say these things:

| What your request says | Why |
|---|---|
| It starts with `$test-driven-development` | Typing the skill's name starts the skill. The skill makes Codex write a test, see it fail, and only then write the code. |
| Write `read_form_1040` in `form1040.py`, as its docstring says, with the tests in `tests/test_form1040.py` | The docstring is the reader's contract: the three keys, whole dollars, and None for a value the return leaves out. |
| The three values you found, as the tests' expected values | "Mistake: copying actual, computed values & pasting them into the expected values of the test." You read them from the return before the code exists. |
| One test reads a copy of the sample return with `TaxableInterestAmt` deleted, written in pytest's `tmp_path`, and expects `taxable_interest` to be None | The schema lets a return leave the element out. ElementTree gives None for an element it does not find, and a reader that reads `.text` of None crashes. |
| Do not commit | You commit in step 4. |

<details><summary>Stuck? One way to say it</summary>

`$test-driven-development Write read_form_1040 in form1040.py, as its docstring says, test first, with the tests in tests/test_form1040.py. In returns/sample-return.xml the expected values are wages 52340, taxable_interest 120 and adjusted_gross_income 52460. Add a test that writes a copy of the sample return without TaxableInterestAmt in tmp_path and expects taxable_interest to be None and the other two values unchanged. Do not commit.`

</details>

You should see Codex run the tests two or more times: first they fail, with `NotImplementedError`, then all pass. If Codex writes the code before a test fails, the check finds it: it runs your tests on a `read_form_1040` that is not written yet. When Codex reports its change, type `/quit`. Then press Enter.

### Step 4. Commit the reader and its tests

Run cell 2 once. It lists what changed in `lab`. `M` marks a changed file that Git tracks. `??` marks a new file that Git does not track yet. Then set `files` to the files that it lists for `form1040.py` and `tests/`. Run the cell again.

| Type | Why |
|---|---|
| `files = "form1040.py tests/test_form1040.py"`, when these are the files that changed | The commit holds the reader and its tests. The skill's files and `skills-lock.json` stay out: the install made them, not the change. |

You should see a line that ends in `Read three Form 1040 values, test first`.

In [ ]:
# 2. Commit the reader and its tests.
files = ""

import subprocess
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if not (lab / ".git").is_dir():
    print("lab is not there yet. Do step 1.")
elif not files:
    print(subprocess.run(["git", "status", "--short"], cwd=lab, capture_output=True, text=True).stdout or "No changes.")
else:
    subprocess.run(["git", "add", *files.split()], cwd=lab)
    done = subprocess.run(["git", "commit", "-m", "Read three Form 1040 values, test first"], cwd=lab, capture_output=True, text=True)
    print(done.stdout.splitlines()[0] if done.returncode == 0 else done.stdout + done.stderr)

### Step 5. Check

Run cell 3. It prints `Done:` when these things are true:

- The skill is in `lab/.agents/skills/test-driven-development`.
- Your tests fail on a `read_form_1040` that is not written yet: they went red.
- All tests pass on your `form1040.py`: they went green.
- One of your tests fails on a reader that crashes when `TaxableInterestAmt` is missing.
- Your reader gives 52340, 120 and 52460 for the sample return, and None for the interest of a return that leaves it out.
- A commit holds the reader and its tests.

Otherwise it prints `Not yet:` and each thing still to do. You do not need to read its code.

In [ ]:
# 3. Check. You do not need to read this code.
%run "{_dh[0]}/check.py"

### Afterwards, together

1. "If you didn't watch the test fail, you don't know if it tests the right thing." What did your red run prove that the green run could not?
2. "Which elements need a missing-element test? Exactly those the schema marks `minOccurs="0"`." Which other element of `return.xsd` would need one, and what should the reader give back?
3. Your test builds the return with no interest in `tmp_path` and leaves `returns/` as it is. Why should a test never change the sample return?

<details><summary>Answer</summary>

The expected values come from the return: `WagesAmt` 52340, `TaxableInterestAmt` 120 and `AdjustedGrossIncomeAmt` 52460, inside `IRS1040`. The missing-element test writes a copy without `TaxableInterestAmt` in `tmp_path` and expects `taxable_interest` to be None, the other two unchanged.

Red, then green: on the unwritten reader, both tests failed with `NotImplementedError`. That shows they call the reader. Then the reader read each value with a full path from the root, and both passed. A reader that reads `.text` of a missing element crashes on the second test. The check tries one.

In the kept run, the agent added a third test of its own: a return whose Form 1040 has no `WagesAmt`. The W-2's `WagesAmt` holds the same number, and a loose path would find it.

The reader and its tests are in [`answers/exercise-03-a-return-reader-test-first`](../answers/exercise-03-a-return-reader-test-first/). Each reason, and where each quoted line comes from, is in [`decisions.md`](../answers/exercise-03-a-return-reader-test-first/decisions.md).

</details>